# 🏛️ JDIH Sumatera Utara — LLM Database Builder v3
> **Fauzan Nur Ahmadi · Universitas Sumatera Utara**
>
> Notebook ini **self-contained** — tidak perlu upload file apapun.
> Jika scraping gagal, corpus demo sintetis otomatis dibuat.

---
### Urutan: jalankan Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8

## Cell 1 — 📦 Install Dependensi

In [ ]:
!pip install -q requests beautifulsoup4 lxml PyMuPDF tqdm rich
print("✅ Dependensi terinstall")


## Cell 2 — 🔗 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUTPUT_DIR = '/content/drive/MyDrive/JDIH_Corpus_Sumut'
os.makedirs(f'{OUTPUT_DIR}/pdf', exist_ok=True)
print(f'✅ Output → {OUTPUT_DIR}')


## Cell 3 — ⚙️ Konfigurasi

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║   KONFIGURASI — edit sesuai kebutuhan           ║
# ╚══════════════════════════════════════════════════╝
import os

# Jika tidak pakai Drive, ganti ke:  OUTPUT_DIR = '/content/jdih_corpus'
OUTPUT_DIR   = '/content/drive/MyDrive/JDIH_Corpus_Sumut'
SUMBER       = 'bpk'          # 'jdih_sumut' | 'bpk' | 'keduanya'
JENIS        = ['perda', 'pergub']
TAHUN_MULAI  = 2015
TAHUN_AKHIR  = 2025
MAKS_HALAMAN = 2              # 1 hal ≈ 20 dokumen
MODE_UJI     = True           # True = maks 3 dok/jenis

os.makedirs(f'{OUTPUT_DIR}/pdf', exist_ok=True)
print('Konfigurasi:')
print(f'  SUMBER={SUMBER}  JENIS={JENIS}')
print(f'  TAHUN={TAHUN_MULAI}-{TAHUN_AKHIR}  MODE_UJI={MODE_UJI}')
print(f'  OUTPUT → {OUTPUT_DIR}')


## Cell 3b — 📚 Load Library (WAJIB DIJALANKAN)
> Jalankan cell ini sebelum Cell 4. Tidak ada yang perlu diedit.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# LIBRARY — semua kelas dan fungsi scraper
# Jalankan cell ini SEBELUM cell lainnya. Tidak ada yang perlu diedit.
# ════════════════════════════════════════════════════════════════════════════
import os, re, json, time, csv, hashlib, logging, urllib.parse
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional

import requests
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

try:
    import fitz  # PyMuPDF — untuk ekstraksi PDF asli
    FITZ_OK = True
except ImportError:
    FITZ_OK = False

log     = logging.getLogger('jdih')
console = Console()

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept': 'text/html,application/xhtml+xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'id-ID,id;q=0.9',
}
DELAY        = 1.5
MAX_RETRY    = 3
CHUNK_SIZE   = 1400
CHUNK_OVERLAP= 150

JENIS_LABEL = {
    'perda': 'Peraturan Daerah Provinsi',
    'pergub': 'Peraturan Gubernur',
    'sk': 'Surat Keputusan',
}

# ── Data model ───────────────────────────────────────────────────────────────
@dataclass
class DokumenHukum:
    id:                 str = ''
    sumber:             str = ''
    jenis:              str = ''
    judul:              str = ''
    nomor:              str = ''
    tahun:              str = ''
    tanggal_ditetapkan: str = ''
    tentang:            str = ''
    status:             str = 'berlaku'
    url_detail:         str = ''
    url_pdf:            str = ''
    path_pdf:           str = ''
    teks_penuh:         str = ''
    sha256_pdf:         str = ''
    jumlah_halaman:     int = 0
    jumlah_karakter:    int = 0
    timestamp_scrape:   str = ''
    error:              str = ''

@dataclass
class ChunkDokumen:
    chunk_id:     str = ''
    doc_id:       str = ''
    jenis:        str = ''
    judul:        str = ''
    nomor:        str = ''
    tahun:        str = ''
    chunk_index:  int = 0
    total_chunks: int = 0
    teks:         str = ''
    karakter:     int = 0

# ── Utilitas ─────────────────────────────────────────────────────────────────
def safe_get(url, session, timeout=30, stream=False):
    for attempt in range(1, MAX_RETRY + 1):
        try:
            r = session.get(url, headers=HEADERS, timeout=timeout, stream=stream)
            r.raise_for_status()
            time.sleep(DELAY)
            return r
        except Exception as e:
            if attempt < MAX_RETRY:
                time.sleep(DELAY * attempt * 2)
    return None

def md5(url):
    return hashlib.md5(url.encode()).hexdigest()[:12]

def sha256f(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for blk in iter(lambda: f.read(65536), b''):
            h.update(blk)
    return h.hexdigest()

def bersihkan(teks):
    teks = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', teks)
    teks = re.sub(r'\n{3,}', '\n\n', teks)
    teks = re.sub(r'[ \t]{2,}', ' ', teks)
    return teks.strip()

def chunking(teks, doc):
    pola = re.compile(
        r'(?=(?:Pasal\s+\d+|BAB\s+[IVXLCDM]+|BAGIAN\s+\w+))',
        re.IGNORECASE)
    segmen = [s.strip() for s in pola.split(teks) if s.strip()]
    raws, buf = [], ''
    for seg in segmen:
        if len(buf) + len(seg) < CHUNK_SIZE:
            buf += '\n' + seg
        else:
            if buf:
                raws.append(buf.strip())
            if len(seg) > CHUNK_SIZE:
                s = 0
                while s < len(seg):
                    raws.append(seg[s:s + CHUNK_SIZE].strip())
                    s += CHUNK_SIZE - CHUNK_OVERLAP
            else:
                buf = seg
    if buf:
        raws.append(buf.strip())
    total = len(raws)
    return [
        ChunkDokumen(
            chunk_id=f'{doc.id}_c{i:04d}', doc_id=doc.id,
            jenis=doc.jenis, judul=doc.judul,
            nomor=doc.nomor, tahun=doc.tahun,
            chunk_index=i, total_chunks=total,
            teks=t, karakter=len(t))
        for i, t in enumerate(raws) if t
    ]

def ekstrak_pdf(path_pdf):
    if not FITZ_OK:
        return '', 0
    try:
        doc = fitz.open(path_pdf)
        hal = len(doc)
        teks = '\n'.join(p.get_text('text') for p in doc)
        doc.close()
        teks = bersihkan(teks)
        if len(teks) < 100:
            print(f'  ⚠️  PDF scan (teks minimal): {Path(path_pdf).name}')
        return teks, hal
    except Exception as e:
        return '', 0

# ── Scraper BPK ──────────────────────────────────────────────────────────────
class BPKScraper:
    BASE = 'https://peraturan.bpk.go.id'

    def __init__(self, session, out_dir):
        self.s   = session
        self.out = Path(out_dir)
        (self.out / 'pdf').mkdir(parents=True, exist_ok=True)

    def crawl(self, jenis='perda', maks=5):
        hasil, hal = [], 1
        print(f'\n🔍 BPK [{jenis.upper()}] crawling...')
        while hal <= maks:
            params = {'kodePropinsi': '12', 'jenis': jenis, 'page': hal}
            url = self.BASE + '/Home/Peraturan?' + urllib.parse.urlencode(params)
            resp = safe_get(url, self.s)
            if resp is None:
                break
            soup = BeautifulSoup(resp.text, 'lxml')
            rows = soup.select('table.table tbody tr')
            if not rows:
                break
            for row in rows:
                cols = row.select('td')
                if len(cols) < 2:
                    continue
                link = cols[1].select_one('a')
                if not link:
                    continue
                doc = DokumenHukum(
                    sumber='bpk', jenis=jenis,
                    timestamp_scrape=datetime.now().isoformat())
                doc.judul = link.get_text(strip=True)
                href = link.get('href', '')
                doc.url_detail = href if href.startswith('http') else self.BASE + href
                doc.id = md5(doc.url_detail)
                m = re.search(r'[Nn]o\.?\s*(\d+)\s*[Tt]ahun\s*(\d{4})', doc.judul)
                if m:
                    doc.nomor, doc.tahun = m.group(1), m.group(2)
                doc.tentang = cols[2].get_text(strip=True) if len(cols) > 2 else ''
                if len(cols) > 3:
                    doc.tanggal_ditetapkan = cols[3].get_text(strip=True)
                hasil.append(doc)
            print(f'  Hal {hal}: +{len(rows)} | total {len(hasil)}')
            if not soup.select_one('a[aria-label="Next"]'):
                break
            hal += 1
        return hasil

    def unduh_pdf(self, doc):
        if not doc.url_detail:
            return doc
        resp = safe_get(doc.url_detail, self.s)
        if resp is None:
            doc.error = 'Gagal akses detail'
            return doc
        soup = BeautifulSoup(resp.text, 'lxml')
        pdf = (soup.select_one('a.btn[href$=".pdf"]') or
               soup.select_one('a[href*="/download/"]') or
               soup.select_one('a[href$=".pdf"]'))
        if not pdf:
            doc.error = 'URL PDF tidak ditemukan'
            return doc
        href = pdf.get('href', '')
        doc.url_pdf = href if href.startswith('http') else self.BASE + href
        nama = f'bpk_{doc.jenis}_{doc.nomor or "x"}_{doc.tahun or "x"}_{doc.id}.pdf'
        path = self.out / 'pdf' / nama
        if not (path.exists() and path.stat().st_size > 1024):
            r = safe_get(doc.url_pdf, self.s, timeout=60, stream=True)
            if r:
                with open(path, 'wb') as f:
                    for chunk in r.iter_content(8192):
                        f.write(chunk)
                print(f'  💾 {nama} ({path.stat().st_size // 1024} KB)')
        if path.exists():
            doc.path_pdf   = str(path)
            doc.sha256_pdf = sha256f(str(path))
        return doc

# ── Scraper JDIH Sumut ───────────────────────────────────────────────────────
class JDIHSumutScraper:
    BASE = 'https://jdih.sumutprov.go.id'

    def __init__(self, session, out_dir):
        self.s   = session
        self.out = Path(out_dir)
        (self.out / 'pdf').mkdir(parents=True, exist_ok=True)

    def crawl(self, jenis='perda', tahun_mulai=2000, tahun_akhir=2025, maks=5):
        hasil, hal = [], 1
        print(f'\n🔍 JDIH Sumut [{jenis.upper()}] crawling...')
        while hal <= maks:
            params = {'jenis': jenis, 'page': hal,
                      'tahun_awal': tahun_mulai, 'tahun_akhir': tahun_akhir}
            url = self.BASE + '/produk-hukum?' + urllib.parse.urlencode(params)
            resp = safe_get(url, self.s)
            if resp is None:
                print(f'  ⚠️  Tidak dapat mengakses halaman {hal}')
                break
            soup = BeautifulSoup(resp.text, 'lxml')
            items = (soup.select('div.produk-hukum-item') or
                     soup.select('table.table tbody tr') or
                     soup.select('div.card'))
            if not items:
                break
            for item in items:
                link = (item.select_one('a.judul') or
                        item.select_one('h4 a') or
                        item.select_one('td:nth-child(2) a') or
                        item.select_one('a'))
                if not link:
                    continue
                doc = DokumenHukum(sumber='jdih_sumut', jenis=jenis,
                                   timestamp_scrape=datetime.now().isoformat())
                doc.judul = link.get_text(strip=True)
                href = link.get('href', '')
                doc.url_detail = href if href.startswith('http') else self.BASE + href
                doc.id = md5(doc.url_detail)
                m = re.search(r'[Nn]o\.?\s*(\d+)\s*[Tt]ahun\s*(\d{4})', doc.judul)
                if m:
                    doc.nomor, doc.tahun = m.group(1), m.group(2)
                hasil.append(doc)
            print(f'  Hal {hal}: +{len(items)} | total {len(hasil)}')
            if not soup.select_one('li.next:not(.disabled), a[rel="next"]'):
                break
            hal += 1
        return hasil

# ── Database builder ─────────────────────────────────────────────────────────
class DatabaseBuilder:
    def __init__(self, out_dir):
        self.out = Path(out_dir)
        self.out.mkdir(parents=True, exist_ok=True)

    def proses_dan_chunk(self, dokumen_list):
        all_chunks = []
        for doc in tqdm(dokumen_list, desc='Chunking'):
            if doc.teks_penuh:
                # Teks sudah ada (demo mode atau dari field langsung)
                doc.jumlah_karakter = len(doc.teks_penuh)
                all_chunks.extend(chunking(doc.teks_penuh, doc))
            elif doc.path_pdf and Path(doc.path_pdf).exists():
                # Ekstrak dari PDF
                teks, hal = ekstrak_pdf(doc.path_pdf)
                doc.teks_penuh      = teks
                doc.jumlah_halaman  = hal
                doc.jumlah_karakter = len(teks)
                if teks:
                    all_chunks.extend(chunking(teks, doc))
            else:
                doc.error = doc.error or 'Tidak ada teks/PDF'
        return all_chunks

    def simpan_semua(self, dokumen_list, chunks):
        # CSV metadata
        fields = ['id','sumber','jenis','judul','nomor','tahun',
                  'tanggal_ditetapkan','tentang','status','url_detail',
                  'url_pdf','path_pdf','sha256_pdf','jumlah_halaman',
                  'jumlah_karakter','timestamp_scrape','error']
        with open(self.out / 'metadata_dokumen.csv', 'w',
                  encoding='utf-8', newline='') as f:
            w = csv.DictWriter(f, fieldnames=fields)
            w.writeheader()
            for doc in dokumen_list:
                row = asdict(doc)
                row.pop('teks_penuh', None)
                w.writerow({k: row.get(k, '') for k in fields})
        print(f'  ✅ metadata_dokumen.csv ({len(dokumen_list)} baris)')

        # JSONL teks penuh
        n = 0
        with open(self.out / 'corpus_teks_penuh.jsonl', 'w', encoding='utf-8') as f:
            for doc in dokumen_list:
                if not doc.teks_penuh:
                    continue
                f.write(json.dumps({
                    'id': doc.id, 'jenis': doc.jenis, 'judul': doc.judul,
                    'nomor': doc.nomor, 'tahun': doc.tahun,
                    'tentang': doc.tentang, 'status': doc.status,
                    'sumber': doc.sumber, 'teks': doc.teks_penuh,
                    'metadata': {'url': doc.url_detail,
                                 'halaman': doc.jumlah_halaman,
                                 'karakter': doc.jumlah_karakter,
                                 'sha256': doc.sha256_pdf,
                                 'timestamp': doc.timestamp_scrape}
                }, ensure_ascii=False) + '\n')
                n += 1
        print(f'  ✅ corpus_teks_penuh.jsonl ({n} dokumen)')

        # JSONL chunks
        with open(self.out / 'corpus_chunks_rag.jsonl', 'w', encoding='utf-8') as f:
            for c in chunks:
                f.write(json.dumps(asdict(c), ensure_ascii=False) + '\n')
        print(f'  ✅ corpus_chunks_rag.jsonl ({len(chunks)} chunks)')

        # Ringkasan
        berhasil = [d for d in dokumen_list if d.teks_penuh]
        per_jenis = {}
        for d in berhasil:
            per_jenis[d.jenis] = per_jenis.get(d.jenis, 0) + 1
        total_kar = sum(d.jumlah_karakter for d in berhasil)
        ringkasan = {
            'timestamp': datetime.now().isoformat(),
            'total_dokumen': len(dokumen_list),
            'berhasil': len(berhasil),
            'gagal': len(dokumen_list) - len(berhasil),
            'per_jenis': per_jenis,
            'total_chunks': len(chunks),
            'total_karakter': total_kar,
            'rata_kar_per_chunk': total_kar // len(chunks) if chunks else 0,
        }
        with open(self.out / 'ringkasan_scraping.json', 'w', encoding='utf-8') as f:
            json.dump(ringkasan, f, ensure_ascii=False, indent=2)
        print(f'  ✅ ringkasan_scraping.json')
        return ringkasan

# ── Banner & ringkasan ────────────────────────────────────────────────────────
def tampilkan_banner():
    console.print(Panel.fit(
        '[bold white]JDIH Sumatera Utara — LLM Database Builder[/bold white]\n'
        '[cyan]Fauzan Nur Ahmadi · Universitas Sumatera Utara[/cyan]',
        border_style='bold blue', padding=(1, 4)))

def tampilkan_ringkasan(r):
    tbl = Table(title='Ringkasan', header_style='bold cyan', border_style='blue')
    tbl.add_column('Metrik', width=28)
    tbl.add_column('Nilai', justify='right', style='bold green')
    tbl.add_row('Total Dokumen',      str(r['total_dokumen']))
    tbl.add_row('Berhasil',           str(r['berhasil']))
    tbl.add_row('Gagal/Scan',         str(r['gagal']))
    tbl.add_row('Total Chunks (RAG)', str(r['total_chunks']))
    tbl.add_row('Total Karakter',     f"{r['total_karakter']:,}")
    tbl.add_row('Rata Kar/Chunk',     f"{r['rata_kar_per_chunk']:,}")
    for j, n in r['per_jenis'].items():
        tbl.add_row(f'  • {JENIS_LABEL.get(j,j)}', str(n))
    console.print(tbl)

print('✅ Library siap. Lanjut ke Cell 3 (Konfigurasi).')


## Cell 4 — 🚀 Scraping + Fallback Demo Otomatis

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 4 — Scraping + Fallback Demo Otomatis
# Jika server tidak merespons, corpus sintetis dibuat secara otomatis.
# ════════════════════════════════════════════════════════════════════════════
import requests as req_lib, hashlib
from datetime import datetime

tampilkan_banner()
session        = req_lib.Session()
semua_dokumen  = []

# ── Coba scraping nyata ───────────────────────────────────────────────────
for jenis in JENIS:
    maks = 1 if MODE_UJI else MAKS_HALAMAN
    print(f'\n{"="*50}  {jenis.upper()}')

    if SUMBER in ('bpk', 'keduanya'):
        try:
            sc = BPKScraper(session, OUTPUT_DIR)
            daftar = sc.crawl(jenis=jenis, maks=maks)
            if MODE_UJI:
                daftar = daftar[:3]
            print(f'  📋 {len(daftar)} dokumen — unduh PDF...')
            for i, doc in enumerate(daftar, 1):
                print(f'  [{i}/{len(daftar)}] {doc.judul[:65]}')
                sc.unduh_pdf(doc)
            semua_dokumen.extend(daftar)
        except Exception as e:
            print(f'  ⚠️  BPK gagal: {e}')

    if SUMBER in ('jdih_sumut', 'keduanya'):
        try:
            sc2 = JDIHSumutScraper(session, OUTPUT_DIR)
            daftar2 = sc2.crawl(
                jenis=jenis,
                tahun_mulai=TAHUN_MULAI,
                tahun_akhir=TAHUN_AKHIR,
                maks=maks)
            if MODE_UJI:
                daftar2 = daftar2[:3]
            semua_dokumen.extend(daftar2)
        except Exception as e:
            print(f'  ⚠️  JDIH Sumut gagal: {e}')

# ── Fallback: corpus demo jika tidak ada dokumen ─────────────────────────
if not semua_dokumen:
    print('\n⚠️  Scraping tidak menghasilkan dokumen.')
    print('   → FALLBACK MODE: corpus sintetis diaktifkan.')
    print('   → Pipeline tetap berjalan penuh untuk pengujian.')

    TEKS_DEMO = {
        'perda_1_2023': (
            'BAB I KETENTUAN UMUM\n\n'
            'Pasal 1\n'
            'Dalam Peraturan Daerah ini yang dimaksud dengan:\n'
            'a. Provinsi adalah Provinsi Sumatera Utara;\n'
            'b. Gubernur adalah Gubernur Sumatera Utara;\n'
            'c. Retribusi Daerah adalah pungutan Daerah sebagai pembayaran '
            'atas jasa atau pemberian izin tertentu yang disediakan '
            'Pemerintah Daerah untuk kepentingan orang pribadi atau badan.\n\n'
            'BAB II OBJEK DAN SUBJEK RETRIBUSI\n\n'
            'Pasal 5\n'
            'Objek Retribusi meliputi:\n'
            'a. Retribusi Jasa Umum;\n'
            'b. Retribusi Jasa Usaha;\n'
            'c. Retribusi Perizinan Tertentu.\n\n'
            'BAB III TARIF RETRIBUSI\n\n'
            'Pasal 10\n'
            'Tarif Retribusi ditetapkan berdasarkan kebijakan daerah dengan '
            'mempertimbangkan biaya penyediaan jasa, kemampuan masyarakat, '
            'dan aspek keadilan.\n\n'
            'BAB IV SANKSI\n\n'
            'Pasal 15\n'
            'Wajib Retribusi yang tidak melaksanakan kewajibannya dikenai '
            'sanksi administratif berupa teguran tertulis dan denda.\n\n'
            'Pasal 16\n'
            'Denda administratif ditetapkan paling banyak '
            'lima puluh juta rupiah.\n\n'
            'BAB V KETENTUAN PENUTUP\n\n'
            'Pasal 20\n'
            'Peraturan Daerah ini mulai berlaku pada tanggal diundangkan.'
        ),
        'perda_3_2022': (
            'BAB I KETENTUAN UMUM\n\n'
            'Pasal 1\n'
            'Lingkungan Hidup adalah kesatuan ruang dengan semua benda, '
            'daya, keadaan, dan makhluk hidup termasuk manusia.\n\n'
            'BAB II KEWENANGAN PROVINSI\n\n'
            'Pasal 5\n'
            'Provinsi berwenang menetapkan kebijakan pengelolaan lingkungan '
            'hidup dan melaksanakan Kajian Lingkungan Hidup Strategis (KLHS).\n\n'
            'BAB III PERIZINAN\n\n'
            'Pasal 10\n'
            'Setiap usaha yang berdampak penting wajib memiliki Amdal.\n\n'
            'BAB IV LARANGAN DAN SANKSI\n\n'
            'Pasal 20\n'
            'Setiap orang dilarang melakukan perbuatan yang mengakibatkan '
            'pencemaran lingkungan hidup.\n\n'
            'Pasal 25\n'
            'Pelanggaran dikenai sanksi administratif berupa teguran '
            'tertulis dan paksaan pemerintah.'
        ),
        'pergub_2_2023': (
            'BAB I KETENTUAN UMUM\n\n'
            'Pasal 1\n'
            'Peraturan Gubernur ini merupakan pelaksanaan Peraturan Daerah '
            'Nomor 1 Tahun 2023 tentang Retribusi Daerah.\n\n'
            'BAB II MEKANISME PEMUNGUTAN\n\n'
            'Pasal 3\n'
            'Pemungutan retribusi dilaksanakan oleh perangkat daerah yang '
            'membidangi urusan pendapatan daerah.\n\n'
            'Pasal 4\n'
            'Wajib retribusi melaksanakan pembayaran melalui loket resmi '
            'atau sistem pembayaran elektronik.\n\n'
            'BAB III PENGELOLAAN PENERIMAAN\n\n'
            'Pasal 8\n'
            'Seluruh hasil pemungutan disetor ke kas daerah paling lambat '
            'satu hari sejak penerimaan.'
        ),
        'perda_5_2021': (
            'BAB I KETENTUAN UMUM\n\n'
            'Pasal 1\n'
            'Ketenagakerjaan adalah segala hal yang berhubungan dengan '
            'tenaga kerja pada waktu sebelum, selama, dan sesudah masa kerja.\n\n'
            'BAB II HAK DAN KEWAJIBAN\n\n'
            'Pasal 5\n'
            'Setiap tenaga kerja berhak atas upah yang layak, perlindungan '
            'keselamatan kerja, dan jaminan sosial.\n\n'
            'Pasal 6\n'
            'Setiap pemberi kerja wajib membayar upah sesuai ketentuan '
            'yang berlaku dan menyediakan fasilitas keselamatan kerja.\n\n'
            'BAB III PENGAWASAN\n\n'
            'Pasal 15\n'
            'Pengawasan ketenagakerjaan dilaksanakan oleh pengawas yang '
            'memiliki kompetensi dan independensi.\n\n'
            'BAB IV PENYELESAIAN PERSELISIHAN\n\n'
            'Pasal 20\n'
            'Perselisihan diselesaikan melalui bipartit, mediasi, atau '
            'Pengadilan Hubungan Industrial.'
        ),
        'perda_7_2020': (
            'BAB I KETENTUAN UMUM\n\n'
            'Pasal 1\n'
            'Pendidikan adalah usaha sadar untuk mewujudkan proses '
            'pembelajaran yang aktif dan bermutu.\n\n'
            'BAB II KEWENANGAN PROVINSI\n\n'
            'Pasal 4\n'
            'Provinsi berwenang mengelola pendidikan menengah dan '
            'pendidikan khusus.\n\n'
            'BAB III PESERTA DIDIK\n\n'
            'Pasal 10\n'
            'Setiap warga negara berhak mendapatkan layanan pendidikan '
            'bermutu tanpa diskriminasi.\n\n'
            'Pasal 11\n'
            'Pemerintah Provinsi wajib menjamin wajib belajar dua belas '
            'tahun tanpa pungutan biaya.\n\n'
            'BAB IV TENAGA PENDIDIK\n\n'
            'Pasal 15\n'
            'Tenaga pendidik wajib memiliki kualifikasi akademik dan '
            'kompetensi sesuai peraturan perundang-undangan.'
        ),
    }

    META_DEMO = [
        ('perda_1_2023', 'perda',  '1',  '2023', 'Retribusi Daerah'),
        ('perda_3_2022', 'perda',  '3',  '2022', 'Pengelolaan Lingkungan Hidup'),
        ('pergub_2_2023','pergub', '2',  '2023', 'Pedoman Pelaksanaan Retribusi'),
        ('perda_5_2021', 'perda',  '5',  '2021', 'Penyelenggaraan Ketenagakerjaan'),
        ('perda_7_2020', 'perda',  '7',  '2020', 'Penyelenggaraan Pendidikan'),
    ]

    for key, jenis, nomor, tahun, tentang in META_DEMO:
        teks = TEKS_DEMO[key]
        judul = (f'Peraturan {("Daerah Provinsi" if jenis=="perda" else "Gubernur")} '
                 f'Sumatera Utara Nomor {nomor} Tahun {tahun} tentang {tentang}')
        doc = DokumenHukum(
            id               = hashlib.md5(key.encode()).hexdigest()[:12],
            sumber           = 'demo',
            jenis            = jenis,
            judul            = judul,
            nomor            = nomor,
            tahun            = tahun,
            tentang          = tentang,
            status           = 'berlaku',
            teks_penuh       = teks,
            jumlah_karakter  = len(teks),
            jumlah_halaman   = max(1, len(teks) // 2000),
            timestamp_scrape = datetime.now().isoformat(),
        )
        semua_dokumen.append(doc)

    print(f'  ✅ {len(semua_dokumen)} dokumen demo dibuat')

print(f'\n✅ Total: {len(semua_dokumen)} dokumen siap diproses')


## Cell 5 — 📄 Ekstraksi, Chunking & Simpan Corpus

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 5 — Ekstraksi, Chunking, dan Simpan Corpus
# ════════════════════════════════════════════════════════════════════════════
builder = DatabaseBuilder(OUTPUT_DIR)

print('🔍 Proses chunking...')
chunks = builder.proses_dan_chunk(semua_dokumen)
print(f'  → {len(chunks)} chunks dihasilkan')

if not chunks:
    print('\n❌ TIDAK ADA CHUNK yang dihasilkan!')
    print('   Kemungkinan penyebab:')
    print('   1. semua_dokumen kosong — jalankan ulang Cell 4')
    print('   2. Semua dokumen PDF scan — jalankan Cell OCR')
    raise ValueError('Corpus kosong — tidak ada chunk')

print('\n💾 Menyimpan corpus...')
ringkasan = builder.simpan_semua(semua_dokumen, chunks)

print()
tampilkan_ringkasan(ringkasan)

# Verifikasi file benar-benar ada
import os
for fname in ['corpus_chunks_rag.jsonl', 'corpus_teks_penuh.jsonl',
              'metadata_dokumen.csv', 'ringkasan_scraping.json']:
    fpath = f'{OUTPUT_DIR}/{fname}'
    if os.path.exists(fpath):
        kb = os.path.getsize(fpath) / 1024
        print(f'  ✅ {fname} ({kb:.1f} KB)')
    else:
        print(f'  ❌ {fname} TIDAK ADA')


## Cell 6 — 🗂️ Pratinjau Corpus

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 6 — Pratinjau Corpus
# ════════════════════════════════════════════════════════════════════════════
import pandas as pd, json, os

csv_path = f'{OUTPUT_DIR}/metadata_dokumen.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f'📊 Total: {len(df)} dokumen')
    print(f'   Teks berhasil  : {(df.jumlah_karakter > 0).sum()}')
    print(f'   Tanpa teks     : {(df.jumlah_karakter == 0).sum()}')
    display(df[['jenis','nomor','tahun','tentang',
                'jumlah_halaman','jumlah_karakter','status']].head(10))

chunks_path = f'{OUTPUT_DIR}/corpus_chunks_rag.jsonl'
if os.path.exists(chunks_path):
    with open(chunks_path, encoding='utf-8') as f:
        semua_chunks = [json.loads(l) for l in f if l.strip()]
    print(f'\n📄 Total chunks: {len(semua_chunks)}')
    print('\nContoh 2 chunk pertama:')
    for i, c in enumerate(semua_chunks[:2]):
        print(f'\n── Chunk {i+1} ──')
        print(f'  ID      : {c["chunk_id"]}')
        print(f'  Dokumen : {c["jenis"].upper()} No.{c["nomor"]}/{c["tahun"]}')
        print(f'  Karakter: {c["karakter"]}')
        print(f'  Teks    : {c["teks"][:250]}...')


## Cell 7 — 🔍 Build FAISS Index
> Aktifkan GPU: `Runtime → Change runtime type → T4 GPU` (opsional)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 7 — Build FAISS Vector Index untuk RAG
# Aktifkan GPU: Runtime → Change runtime type → T4 GPU (opsional)
# ════════════════════════════════════════════════════════════════════════════
import os, json, pickle
import numpy as np
from pathlib import Path

CHUNKS_PATH     = f'{OUTPUT_DIR}/corpus_chunks_rag.jsonl'
INDEX_DIR       = f'{OUTPUT_DIR}/faiss_index'
MODEL_EMBEDDING = 'paraphrase-multilingual-MiniLM-L12-v2'
# Model lebih akurat (unduh ~400MB):
# MODEL_EMBEDDING = 'LazarusNLP/IndoNanoSE-Base'

# Guard
if not os.path.exists(CHUNKS_PATH):
    raise FileNotFoundError(
        f'File tidak ditemukan: {CHUNKS_PATH}\n'
        'Pastikan Cell 5 (Ekstraksi) sudah dijalankan berhasil.'
    )

!pip install -q sentence-transformers faiss-cpu

import faiss
from sentence_transformers import SentenceTransformer

Path(INDEX_DIR).mkdir(parents=True, exist_ok=True)

with open(CHUNKS_PATH, encoding='utf-8') as f:
    chunks = [json.loads(l) for l in f if l.strip()]
print(f'✅ {len(chunks)} chunks dimuat')

teks_list = [c['teks'] for c in chunks]
print(f'⏳ Embedding dengan {MODEL_EMBEDDING}...')
model = SentenceTransformer(MODEL_EMBEDDING)
embeddings = model.encode(
    teks_list, batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
).astype('float32')
dim = embeddings.shape[1]
print(f'✅ Embedding selesai — dimensi: {dim}')

if len(chunks) < 10000:
    index = faiss.IndexFlatIP(dim)
else:
    nlist = min(100, len(chunks) // 39)
    q = faiss.IndexFlatIP(dim)
    index = faiss.IndexIVFFlat(q, dim, nlist, faiss.METRIC_INNER_PRODUCT)
    index.train(embeddings)
index.add(embeddings)
print(f'✅ FAISS index: {index.ntotal} vektor')

faiss.write_index(index, f'{INDEX_DIR}/index.faiss')

with open(f'{INDEX_DIR}/metadata.pkl', 'wb') as f:
    pickle.dump([{
        'chunk_id': c['chunk_id'], 'doc_id': c['doc_id'],
        'jenis': c['jenis'], 'judul': c['judul'],
        'nomor': c['nomor'], 'tahun': c['tahun'],
        'teks_preview': c['teks'][:300]
    } for c in chunks], f)

with open(f'{INDEX_DIR}/config.json', 'w') as f:
    json.dump({'model': MODEL_EMBEDDING, 'dim': dim,
               'total': len(chunks)}, f, indent=2)

print(f'\n✅ Index tersimpan → {INDEX_DIR}')
!ls -lh {INDEX_DIR}


## Cell 8 — 🧪 Test Retrieval

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 8 — Test Retrieval
# ════════════════════════════════════════════════════════════════════════════
import os, faiss, pickle, json
import numpy as np

INDEX_DIR = f'{OUTPUT_DIR}/faiss_index'

# Guard
missing = [fn for fn in ['index.faiss','metadata.pkl','config.json']
           if not os.path.exists(f'{INDEX_DIR}/{fn}')]
if missing:
    isi = os.listdir(INDEX_DIR) if os.path.exists(INDEX_DIR) else ['(folder kosong)']
    raise FileNotFoundError(
        f'File index hilang: {missing}\n'
        f'Isi folder saat ini: {isi}\n'
        'Pastikan Cell 7 sudah dijalankan hingga selesai.'
    )

from sentence_transformers import SentenceTransformer

with open(f'{INDEX_DIR}/config.json') as f:
    cfg = json.load(f)
with open(f'{INDEX_DIR}/metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

index     = faiss.read_index(f'{INDEX_DIR}/index.faiss')
model_ret = SentenceTransformer(cfg['model'])
print(f'✅ Retriever siap — {cfg["total"]} chunks | model: {cfg["model"]}')

# ── Ganti query sesuai kebutuhan ──────────────────────────────────────────
QUERY = 'sanksi pidana pelanggaran retribusi daerah'
TOP_K = 5

vec = model_ret.encode([QUERY], normalize_embeddings=True).astype('float32')
scores, indices = index.search(vec, TOP_K)

print(f'\n🔎 Query: "{QUERY}"')
print('=' * 65)
for i, (skor, idx) in enumerate(zip(scores[0], indices[0]), 1):
    if idx < 0:
        continue
    m = metadata[idx]
    print(f'\n  [{i}] Relevansi : {skor:.4f}')
    print(f'       Dokumen  : {m["jenis"].upper()} No.{m["nomor"]}/{m["tahun"]}')
    print(f'       Preview  : {m["teks_preview"][:200]}...')


## Cell 9 — 💾 Download File ke Komputer

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 9 — Download File ke Komputer
# ════════════════════════════════════════════════════════════════════════════
from google.colab import files
import os

for fname in ['metadata_dokumen.csv', 'corpus_teks_penuh.jsonl',
              'corpus_chunks_rag.jsonl', 'ringkasan_scraping.json']:
    fpath = f'{OUTPUT_DIR}/{fname}'
    if os.path.exists(fpath):
        print(f'📥 {fname} ({os.path.getsize(fpath)//1024} KB)')
        files.download(fpath)
    else:
        print(f'⚠️  Tidak ada: {fname}')


## Cell 10 — 🔤 OCR untuk PDF Scan (Opsional)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 10 — OCR untuk PDF Scan (Opsional)
# Jalankan jika banyak dokumen dengan jumlah_karakter = 0
# ════════════════════════════════════════════════════════════════════════════
!apt-get install -qq tesseract-ocr tesseract-ocr-ind
!pip install -q pytesseract pdf2image Pillow

import pytesseract
from pdf2image import convert_from_path
from pathlib import Path

def ocr_pdf(path_pdf, bahasa='ind'):
    gambar = convert_from_path(path_pdf, dpi=200)
    return bersihkan('\n'.join(
        pytesseract.image_to_string(g, lang=bahasa) for g in gambar))

# Proses ulang dokumen dengan teks kosong
scan_docs = [d for d in semua_dokumen
             if d.jumlah_karakter == 0 and d.path_pdf
             and Path(d.path_pdf).exists()]
print(f'PDF scan terdeteksi: {len(scan_docs)}')

if scan_docs:
    for doc in tqdm(scan_docs, desc='OCR'):
        teks = ocr_pdf(doc.path_pdf)
        doc.teks_penuh      = teks
        doc.jumlah_karakter = len(teks)
        print(f'  ✅ {doc.judul[:55]} → {len(teks):,} karakter')

    # Simpan ulang dengan hasil OCR
    builder2 = DatabaseBuilder(OUTPUT_DIR)
    chunks_baru = builder2.proses_dan_chunk(scan_docs)
    print(f'  +{len(chunks_baru)} chunks baru dari OCR')
    builder2.simpan_semua(semua_dokumen,
                          [c for d in semua_dokumen
                           for c in (chunking(d.teks_penuh, d) if d.teks_penuh else [])])
    print('✅ Corpus diperbarui dengan hasil OCR')
else:
    print('Tidak ada PDF scan — OCR tidak diperlukan.')
